# Práctica 4: Análisis de Logs de Seguridad (SIEM)

TODO: Agregar contexto y portada

## Fase 1 - Diagnóstico del Dataset

### 1.1 Carga del dataset y tipos
Carga el dataset. Muestra `shape`, `dtypes` e `info()`. ¿Cuántas columnas tienen tipo incorrecto?

**Apartado de código:** usar la celda siguiente.

In [1]:
from pathlib import Path

import pandas as pd

import utils

DATASET_PATH = Path("content/siem_eventos_sucio - siem_eventos_sucio.csv")

df_raw = utils.load_siem_dataset(DATASET_PATH)
df = df_raw.copy()

display(df.head())
utils.styled_output(str(df.shape), "Shape del dataset")
utils.styled_output(df.dtypes.astype(str).to_string(), "dtypes actuales")
utils.styled_output(utils.dataframe_info_text(df), "info()")

expected_dtypes = {
    "timestamp_evento": "datetime64[ns]",
    "timestamp_resolucion": "datetime64[ns]",
    "puerto_destino": "Int64",
}
incorrect_type_report = [
    f"- {column}: esperado {expected_dtype}, actual {df[column].dtype}"
    for column, expected_dtype in expected_dtypes.items()
]
utils.styled_output(
    "\n".join(incorrect_type_report),
    f"Columnas con tipo incorrecto: {len(incorrect_type_report)}",
)

,evento_id,timestamp_evento,timestamp_resolucion,ip_origen,ip_destino,puerto_destino,protocolo,tipo_evento,categoria,severidad,...,sistema_operativo,pais_origen,usuario,analista_id,bytes_enviados,bytes_recibidos,accion_tomada,tiempo_respuesta_min,falso_positivo,resuelto
0,EVT101614,2024-08-23 21:11:26,2024-08-23 22:07:26,151.70.241.104,10.9.34.76,3306.0,FTP,Phishing detectado,Ingeniería social,Media,...,Windows 11 Pro,México,user_martinez11,ANA008,1036785,280295,Cuarentenado,56,False,True
1,EVT104241,03/16/2024 13:18,2024-03-16 11:53:00,66.238.112.228,10.8.47.206,8443.0,SMB,Exfiltración de datos,Exfiltración,Crítica,...,Linux Debian 11,Irán,user_garcia88,ANA009,1918856,949645,Cuarentenado,50,False,True
2,EVT104669,2024-10-30 5:31:03,NaN,213.2.168.63,10.3.13.216,3389.0,RDP,Ransomware detectado,Malware,Crítica,...,Linux Ubuntu 22.04,China,user_gonzalez57,ANA010,117858,1230502,Alertado,28,False,True
3,EVT102694,2024-02-23 13:05:27,2024-02-23 15:13:27,167.175.67.104,10.0.43.59,80.0,TCP,Escalación de privilegios,Control de acceso,Alta,...,Cisco IOS,Brasil,NaN,ANA008,3326046,1756088,Bloqueado,128,False,True
4,EVT104256,2024-05-28 7:24:59,2024-05-28 9:58:59,45.60.51.96,10.2.43.122,3306.0,HTTPS,Acceso no autorizado,Control de acceso,Alta,...,Windows Server 2022,Estados Unidos,NaN,ANA010,1309398,1415415,Bloqueado,154,False,True


# Shape del dataset
(5515, 21)
# dtypes actuales
evento_id                   str
timestamp_evento            str
timestamp_resolucion        str
ip_origen                   str
ip_destino                  str
puerto_destino          float64
protocolo                   str
tipo_evento                 str
categoria                   str
severidad                   str
sistema_afectado            str
sistema_operativo           str
pais_origen                 str
usuario                     str
analista_id                 str
bytes_enviados            int64
bytes_recibidos           int64
accion_tomada               str
tiempo_respuesta_min      int64
falso_positivo             bool
resuelto                   bool
# info()
<class 'pandas.DataFrame'>
RangeIndex: 5515 entries, 0 to 5514
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   evento_id             5515 non-null   str    
 1   timestamp_evento

### 1.2 Nulos por columna
Calcula el porcentaje de nulos por columna. Identifica cuáles nulos son esperados y lógicos.

**Apartado de código:** usar la celda siguiente.

In [2]:
null_summary = utils.null_percentage_summary(df)
display(null_summary.query("nulos > 0"))

expected_nulls = [
    "- timestamp_resolucion: es normal cuando el evento sigue abierto.",
    "- usuario: muchos eventos de red no están asociados a una cuenta concreta.",
]
unexpected_nulls = [
    "- analista_id: deja eventos sin responsable asignado.",
    "- pais_origen: reduce la calidad del análisis geográfico.",
    "- puerto_destino: impide validar con precisión el servicio objetivo.",
    "- sistema_operativo: complica el análisis por plataforma.",
]
utils.styled_output(
    "\n".join(expected_nulls + [""] + unexpected_nulls),
    "Interpretación de nulos",
)

,nulos,porcentaje
timestamp_resolucion,1021,18.51
usuario,864,15.67
analista_id,20,0.36
pais_origen,15,0.27
puerto_destino,12,0.22
sistema_operativo,10,0.18


# Interpretación de nulos
- timestamp_resolucion: es normal cuando el evento sigue abierto.
- usuario: muchos eventos de red no están asociados a una cuenta concreta.

- analista_id: deja eventos sin responsable asignado.
- pais_origen: reduce la calidad del análisis geográfico.
- puerto_destino: impide validar con precisión el servicio objetivo.
- sistema_operativo: complica el análisis por plataforma.


### 1.3 Valores únicos en columnas categóricas
Muestra valores únicos de: `tipo_evento`, `severidad`, `accion_tomada`, `categoria`, `pais_origen`.

**Apartado de código:** usar la celda siguiente.

In [3]:
categorical_columns = [
    "tipo_evento",
    "severidad",
    "accion_tomada",
    "categoria",
    "pais_origen",
]

for column in categorical_columns:
    utils.styled_output(column, "Columna analizada")
    display(utils.top_value_counts(df, column, dropna=False, top_n=20))

# Columna analizada
tipo_evento


,conteo
tipo_evento,
Ransomware detectado,381
Escalación de privilegios,381
Exfiltración de datos,378
XSS detectado,372
Fuerza bruta,366
Escaneo de red,360
DDoS,358
Phishing detectado,356
Login exitoso,352


# Columna analizada
severidad


,conteo
severidad,
Crítica,1661
Alta,1509
Media,1396
Baja,811
ALTA,26
CRÍTICA,23
crítica,16
alta,13
critica,12


# Columna analizada
accion_tomada


,conteo
accion_tomada,
Alertado,1482
Bloqueado,1264
Cuarentenado,1162
En revisión,1146
Permitido,351
BLOQUEADO,21
bloqueado,17
EN REVISIÓN,12
CUARENTENADO,11


# Columna analizada
categoria


,conteo
categoria,
Autenticación,1063
Control de acceso,731
Web,723
Malware,720
Reconocimiento,714
Exfiltración,381
Disponibilidad,365
Ingeniería social,363
Post-explotación,355


# Columna analizada
pais_origen


,conteo
pais_origen,
México,1743
India,356
Irán,355
Holanda,354
Ucrania,351
Corea del Norte,350
Rusia,336
China,327
Alemania,321


### 1.4 Validación visual de `ip_origen`
Detecta visualmente IPs con formato inválido de IPv4.

**Apartado de código:** usar la celda siguiente.

In [4]:
invalid_ip_examples = utils.invalid_ipv4_examples(df["ip_origen"])
utils.styled_output(
    "\n".join(f"- {ip}" for ip in invalid_ip_examples[:10]),
    f"Ejemplos de IPs inválidas detectadas: {len(invalid_ip_examples)}",
)

display(
    df.loc[df["ip_origen"].astype(str).isin(invalid_ip_examples), ["evento_id", "ip_origen"]]
    .drop_duplicates()
    .head(10)
)

# Ejemplos de IPs inválidas detectadas: 5
- 192.168.1
- 10.0.0.999
- 256.1.1.1
- 10.0..5
- abc.def.ghi.jkl


,evento_id,ip_origen
410,EVT103285,192.168.1
1013,EVT101126,10.0.0.999
1723,EVT101162,192.168.1
2047,EVT101697,256.1.1.1
2491,EVT101659,10.0..5
2511,EVT100839,10.0.0.999
2552,EVT101089,abc.def.ghi.jkl
2557,EVT100930,192.168.1
2747,EVT103548,10.0.0.999
3736,EVT102003,192.168.1


### 1.5 Estadística descriptiva de numéricas
Aplica `describe()` a columnas numéricas e identifica columnas con valores físicamente imposibles (negativos, etc.).

**Apartado de código:** usar la celda siguiente.

In [5]:
numeric_columns = [
    "puerto_destino",
    "bytes_enviados",
    "bytes_recibidos",
    "tiempo_respuesta_min",
]

display(df[numeric_columns].describe().T)

impossible_values_summary = pd.DataFrame(
    {
        "regla": [
            "puerto_destino <= 0",
            "puerto_destino > 65535",
            "bytes_enviados < 0",
            "bytes_recibidos < 0",
            "tiempo_respuesta_min <= 0",
        ],
        "casos": [
            int((pd.to_numeric(df["puerto_destino"], errors="coerce") <= 0).sum()),
            int((pd.to_numeric(df["puerto_destino"], errors="coerce") > 65535).sum()),
            int((pd.to_numeric(df["bytes_enviados"], errors="coerce") < 0).sum()),
            int((pd.to_numeric(df["bytes_recibidos"], errors="coerce") < 0).sum()),
            int((pd.to_numeric(df["tiempo_respuesta_min"], errors="coerce") <= 0).sum()),
        ],
    }
)
display(impossible_values_summary)
utils.styled_output(
    "Las columnas que no deberían tener valores negativos son puerto_destino, bytes_enviados, bytes_recibidos y tiempo_respuesta_min.",
    "Lectura del describe()",
)

,count,mean,std,min,25%,50%,75%,max
puerto_destino,5503.0,2.293848e+03,4.637248e+03,-1.0,25.0,443.0,3389.0,100000.0
bytes_enviados,5515.0,2.566948e+06,2.134233e+06,-500.0,1219062.5,2495767.0,3743576.5,40809824.0
bytes_recibidos,5515.0,1.004274e+06,5.761712e+05,458.0,512422.5,991216.0,1507670.0,1999871.0
tiempo_respuesta_min,5515.0,2.992718e+02,7.574097e+02,-10.0,39.0,103.0,284.0,9993.0


,regla,casos
0,puerto_destino <= 0,2
1,puerto_destino > 65535,8
2,bytes_enviados < 0,6
3,bytes_recibidos < 0,0
4,tiempo_respuesta_min <= 0,6


# Lectura del describe()
Las columnas que no deberían tener valores negativos son puerto_destino, bytes_enviados, bytes_recibidos y tiempo_respuesta_min.


### 1.6 Lista de problemas detectados

- Hay 3 columnas con tipo claramente incorrecto para el análisis: `timestamp_evento`, `timestamp_resolucion` y `puerto_destino`.
- Existen nulos lógicos en `timestamp_resolucion` y `usuario`, porque un evento puede seguir abierto o no estar asociado a una cuenta.
- También hay nulos no ideales en `analista_id`, `pais_origen`, `puerto_destino` y `sistema_operativo`.
- Las columnas categóricas tienen ruido textual: mayúsculas inconsistentes, espacios y variantes duplicadas del mismo valor.
- `severidad` contiene sinónimos no estándar como `Grave` y `Urgente`.
- `accion_tomada` incluye valores fuera del catálogo esperado, por ejemplo `Ignorado` y `Cuarentena`.
- `pais_origen` mezcla sinónimos para el mismo país, como `Russia`/`Rusia` y `EEUU`/`United States`/`Estados Unidos`.
- `ip_origen` contiene direcciones con formato IPv4 inválido, por ejemplo `192.168.1`, `abc.def.ghi.jkl`, `10.0.0.999` y `256.1.1.1`.
- En variables numéricas hay valores físicamente imposibles: puertos fuera de rango, `bytes_enviados` negativos y `tiempo_respuesta_min` menor o igual a cero.
- El dataset contiene 15 duplicados exactos que deben eliminarse en la limpieza.

## Fase 2 - Limpieza de Texto y Estandarización

### 2.1 Normalización de texto
Aplica normalización (`strip()`, `title()`) a: `tipo_evento`, `severidad`, `accion_tomada`, `categoria`, `pais_origen`, `protocolo`, `sistema_operativo`.

**Apartado de código:** usar la celda siguiente.

In [6]:
# Reiniciamos desde el dataset original para separar diagnóstico y limpieza.
df = df_raw.copy()

text_columns = [
    "tipo_evento",
    "severidad",
    "accion_tomada",
    "categoria",
    "pais_origen",
    "protocolo",
    "sistema_operativo",
]

df = utils.normalize_text_columns(df, text_columns)

# Correcciones puntuales para etiquetas técnicas que title() no deja en el formato deseado.
df["tipo_evento"] = df["tipo_evento"].replace(
    {
        "Ddos": "DDoS",
        "Xss Detectado": "XSS Detectado",
        "Inyección Sql": "Inyección SQL",
    }
)
df["protocolo"] = df["protocolo"].str.upper()

display(df[text_columns].head())

,tipo_evento,severidad,accion_tomada,categoria,pais_origen,protocolo,sistema_operativo
0,Phishing Detectado,Media,Cuarentenado,Ingeniería Social,México,FTP,Windows 11 Pro
1,Exfiltración De Datos,Crítica,Cuarentenado,Exfiltración,Irán,SMB,Linux Debian 11
2,Ransomware Detectado,Crítica,Alertado,Malware,China,RDP,Linux Ubuntu 22.04
3,Escalación De Privilegios,Alta,Bloqueado,Control De Acceso,Brasil,TCP,Cisco Ios
4,Acceso No Autorizado,Alta,Bloqueado,Control De Acceso,Estados Unidos,HTTPS,Windows Server 2022


### 2.2 Estandarización de severidad
Deja solo: `Crítica`, `Alta`, `Media`, `Baja`. Define y documenta mapeo de `Grave` y `Urgente`.

**Apartado de código:** usar la celda siguiente.

In [7]:
# Decisión de negocio: 'Grave' y 'Urgente' se homologan a 'Alta'
# porque expresan riesgo elevado, pero no forman parte del catálogo oficial del SOC.
severity_map = {
    "Critica": "Crítica",
    "Crítica": "Crítica",
    "Cr�tica": "Crítica",
    "Grave": "Alta",
    "Urgente": "Alta",
}
df["severidad"] = df["severidad"].replace(severity_map)

display(utils.top_value_counts(df, "severidad", dropna=False))

unexpected_severity = sorted(set(df["severidad"].dropna()) - {"Crítica", "Alta", "Media", "Baja"})
utils.styled_output(
    "Valores fuera del catálogo: " + (", ".join(unexpected_severity) if unexpected_severity else "ninguno"),
    "Validación de severidad",
)

,conteo
severidad,
Crítica,1712
Alta,1561
Media,1418
Baja,824


# Validación de severidad
Valores fuera del catálogo: ninguno


### 2.3 Estandarización de `accion_tomada`
Catálogo objetivo: `Bloqueado`, `Permitido`, `En Revisión`, `Alertado`, `Cuarentenado`. Define qué hacer con `Ignorado`.

**Apartado de código:** usar la celda siguiente.

In [8]:
# Decisión de negocio: 'Ignorado' se homologa a 'Permitido'
# porque implica que no hubo contención real sobre el evento.
action_map = {
    "En Revision": "En Revisión",
    "En Revisi�n": "En Revisión",
    "Cuarentena": "Cuarentenado",
    "Ignorado": "Permitido",
}
df["accion_tomada"] = df["accion_tomada"].replace(action_map)

display(utils.top_value_counts(df, "accion_tomada", dropna=False))

expected_actions = {"Bloqueado", "Permitido", "En Revisión", "Alertado", "Cuarentenado"}
unexpected_actions = sorted(set(df["accion_tomada"].dropna()) - expected_actions)
utils.styled_output(
    "Valores fuera del catálogo: " + (", ".join(unexpected_actions) if unexpected_actions else "ninguno"),
    "Validación de accion_tomada",
)

,conteo
accion_tomada,
Alertado,1497
Bloqueado,1302
Cuarentenado,1189
En Revisión,1165
Permitido,362


# Validación de accion_tomada
Valores fuera del catálogo: ninguno


### 2.4 Unificación de `pais_origen`
Unifica sinónimos: `Russia`/`Rusia`; `EEUU`/`United States`/`Estados Unidos`.

**Apartado de código:** usar la celda siguiente.

In [9]:
country_map = {
    "Russia": "Rusia",
    "Eeuu": "Estados Unidos",
    "United States": "Estados Unidos",
    "M�xico": "México",
    "Ir�n": "Irán",
    "Corea Del Norte": "Corea del Norte",
}
df["pais_origen"] = df["pais_origen"].replace(country_map)

display(utils.top_value_counts(df, "pais_origen", dropna=False, top_n=15))

,conteo
pais_origen,
México,1758
India,362
Irán,359
Holanda,356
Ucrania,356
Corea del Norte,352
Rusia,349
China,335
Estados Unidos,333


### 2.5 Duplicados exactos
Detecta y elimina duplicados exactos. Reporta cuántos se eliminaron.

**Apartado de código:** usar la celda siguiente.

In [10]:
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

utils.styled_output(
    f"Duplicados exactos detectados: {duplicates_before}\nFilas después de eliminarlos: {len(df)}",
    "Resultado de deduplicación",
)

# Resultado de deduplicación
Duplicados exactos detectados: 15
Filas después de eliminarlos: 5500


## Fase 3 - Timestamps y Tipos de Datos

### 3.1 Conversión de `timestamp_evento`
Convierte a datetime considerando mezcla de formatos (incluye MM/DD/YYYY).

**Apartado de código:** usar la celda siguiente.

### 3.2 Conversión de `timestamp_resolucion`
Convierte a datetime y explica por qué no usar `errors='coerce'` sin análisis previo.

**Apartado de código y justificación:** usar celdas siguientes.

**Justificación 3.2:** escribe aquí tu explicación.

### 3.3 Conversión de columnas numéricas
Convierte `bytes_enviados`, `bytes_recibidos`, `puerto_destino`, `tiempo_respuesta_min` a enteros, manejando nulos con `Int64`.

**Apartado de código:** usar la celda siguiente.

### 3.4 Conversión de booleanos
Convierte `falso_positivo` y `resuelto` a tipo booleano.

**Apartado de código:** usar la celda siguiente.

## Fase 4 - Valores Imposibles y Outliers

### 4.1 Validación de puertos
Detecta y elimina puertos fuera del rango 1-65535.

**Apartado de código:** usar la celda siguiente.

### 4.2 Valores negativos o cero
Detecta filas con `bytes_enviados` o `tiempo_respuesta_min` negativos o cero y justifica la decisión de tratamiento.

**Apartado de código y justificación:** usar celdas siguientes.

**Justificación 4.2:** escribe aquí tu decisión.

### 4.3 Validación de IPv4
Valida `ip_origen` (4 octetos, cada uno entre 0 y 255) y detecta IPs inválidas con Python.

**Apartado de código:** usar la celda siguiente.

## Fase 5 - Inconsistencias Lógicas (Nivel SOC)

### 5.1 Falso positivo bloqueado
Detecta casos donde `falso_positivo = True` y `accion_tomada` sea `Bloqueado` o `Cuarentenado`. Decide qué campo corregir.

**Apartado de código:** usar la celda siguiente.

### 5.2 Resuelto sin timestamp
Detecta casos con `resuelto = True` y `timestamp_resolucion` nulo.

**Apartado de código:** usar la celda siguiente.

### 5.3 Resolución antes del evento
Detecta y elimina filas donde `timestamp_resolucion < timestamp_evento`.

**Apartado de código:** usar la celda siguiente.

### 5.4 Tiempo de respuesta inconsistente
Crea `tiempo_calculado` desde timestamps y compara con `tiempo_respuesta_min`. Cuenta diferencias mayores a 30 minutos.

**Apartado de código:** usar la celda siguiente.

### 5.5 Violación de SLA
Aplica reglas SLA por severidad. Reporta porcentaje de Críticos > 60 min y Altos > 240 min.

**Apartado de código:** usar la celda siguiente.

### 5.6 IP origen igual a IP destino
Detecta y elimina filas donde `ip_origen == ip_destino`.

**Apartado de código:** usar la celda siguiente.

### 5.7 Login exitoso bloqueado
Detecta casos de `tipo_evento = 'Login Exitoso'` con `accion_tomada = 'Bloqueado'`.

**Apartado de código:** usar la celda siguiente.

### 5.8 FP crítico no resuelto
Detecta registros con `falso_positivo = True`, `severidad = 'Crítica'` y `resuelto = False`. Interpreta qué indica sobre triage.

**Apartado de código:** usar la celda siguiente.

## Fase 6 - Imputación y Decisiones Finales

### 6.1 Nulos en `usuario`
Decide si imputar o dejar nulos y justifica según el contexto de seguridad.

**Apartado de decisión (markdown) y código (si aplica):** usar celdas siguientes.

**Justificación 6.1:** escribe aquí tu criterio.

### 6.2 Nulos en `pais_origen`
Decide entre imputar con `Desconocido` o eliminar filas y explica impacto en el análisis.

**Apartado de decisión (markdown) y código (si aplica):** usar celdas siguientes.

**Justificación 6.2:** escribe aquí tu criterio.

### 6.3 Nulos en `puerto_destino`
Evalúa si imputar (ej. mediana) tiene sentido o si debe conservarse como ausencia de dato.

**Apartado de decisión (markdown) y código (si aplica):** usar celdas siguientes.

**Justificación 6.3:** escribe aquí tu criterio.

### 6.4 Aplicación de decisiones finales
Aplica tus decisiones y reporta cuántos nulos quedan al final.

**Apartado de código:** usar la celda siguiente.

## Fase 7 - Análisis y Visualización

### 7.1 Tasa de falsos positivos por tipo de evento
Calcula la métrica `tasa_fp` por `tipo_evento`.

**Apartado de código:** usar la celda siguiente.

### 7.2 Gráfica de tasa de FP
Crea una barra horizontal ordenada, con línea vertical punteada del promedio global.

**Apartado de código (visualización 1):** usar la celda siguiente.

### 7.3 Críticos sin resolver por sistema operativo
Filtra `severidad = 'Crítica'` y `resuelto = False`, agrupa por `sistema_operativo` y cuenta.

**Apartado de código:** usar la celda siguiente.

### 7.4 Segunda gráfica por sistema operativo
Genera una segunda gráfica (barras o treemap) para los SO con más críticos sin resolver (usa rojo o escala de calor).

**Apartado de código (visualización 2):** usar la celda siguiente.

### 7.5 Conclusión final para el SOC
Responde ambas preguntas del SOC con números concretos e incluye cuántos eventos violaron el SLA.

**Apartado de conclusión:** completar esta celda markdown.

## Criterios de Evaluación (Referencia)
- Diagnóstico documentado: 10
- Limpieza de texto: 15
- Timestamps y tipos: 10
- Valores imposibles: 10
- Inconsistencias lógicas: 30
- Imputación de nulos: 10
- Visualizaciones: 10
- Conclusión del SOC: 5
- Total: 100